In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
train_df = pd.read_csv('../data/train.csv')

# Definisikan kolom kategorikal secara lengkap
categorical_cols = ['gender', 'country', 'device_type', 'device_os', 'merchant_category', 'transaction_type']

print(f"Kolom kategorikal yang dianalisis: {categorical_cols}")
print(f"\nTotal data: {len(train_df)}")
print(f"Total fraud (is_fraud=1): {train_df['is_fraud'].sum()}")
print(f"Total non-fraud (is_fraud=0): {(train_df['is_fraud']==0).sum()}")

# Tampilkan semua kategori unik per kolom
print("\n" + "=" * 70)
print("DAFTAR KATEGORI UNIK PER KOLOM")
print("=" * 70)
for col in categorical_cols:
    unique_vals = train_df[col].unique()
    print(f"\n{col} ({len(unique_vals)} kategori):")
    print(f"  {list(unique_vals)}")

Kolom kategorikal yang dianalisis: ['gender', 'country', 'device_type', 'device_os', 'merchant_category', 'transaction_type']

Total data: 100000
Total fraud (is_fraud=1): 14128
Total non-fraud (is_fraud=0): 85872

DAFTAR KATEGORI UNIK PER KOLOM

gender (2 kategori):
  ['M', 'F']

country (10 kategori):
  ['UK', 'IN', 'US', 'ID', 'BR', 'CA', 'DE', 'AU', 'FR', 'SG']

device_type (3 kategori):
  ['mobile', 'desktop', 'tablet']

device_os (5 kategori):
  ['Android', 'iOS', 'Windows', 'Linux', 'MacOS']

merchant_category (9 kategori):
  ['clothing', 'restaurants', 'electronics', 'gas', 'services', 'groceries', 'travel', 'financial', 'luxury']

transaction_type (4 kategori):
  ['purchase', 'topup', 'withdrawal', 'transfer']


In [9]:
# Hitung jumlah is_fraud per masing-masing kategori
print("=" * 70)
print("JUMLAH FRAUD (is_fraud=1) PER KATEGORI")
print("=" * 70)

for col in categorical_cols:
    print(f"\n{'='*50}")
    print(f"Kolom: {col}")
    print(f"{'='*50}")
    
    # Crosstab untuk melihat distribusi fraud per kategori
    crosstab = pd.crosstab(train_df[col], train_df['is_fraud'], margins=True)
    crosstab.columns = ['Non-Fraud (0)', 'Fraud (1)', 'Total']
    
    # Hitung persentase fraud per kategori
    crosstab['Fraud Rate (%)'] = (crosstab['Fraud (1)'] / crosstab['Total'] * 100).round(2)
    
    print(crosstab)
    print()

JUMLAH FRAUD (is_fraud=1) PER KATEGORI

Kolom: gender
        Non-Fraud (0)  Fraud (1)   Total  Fraud Rate (%)
gender                                                  
F               35336       5801   41137           14.10
M               50536       8327   58863           14.15
All             85872      14128  100000           14.13


Kolom: country
         Non-Fraud (0)  Fraud (1)   Total  Fraud Rate (%)
country                                                  
AU                 400         50     450           11.11
BR                2214        387    2601           14.88
CA                9652       1569   11221           13.98
DE                1253        209    1462           14.30
FR                1534        232    1766           13.14
ID               10370       1694   12064           14.04
IN               25229       4146   29375           14.11
SG                 438         57     495           11.52
UK                9260       1575   10835           14.54
US    

In [10]:
# Ringkasan: Kategori dengan fraud rate tertinggi
print("=" * 70)
print("RINGKASAN: KATEGORI DENGAN FRAUD RATE TERTINGGI")
print("=" * 70)

for col in categorical_cols:
    fraud_rate = train_df.groupby(col)['is_fraud'].mean() * 100
    top_fraud = fraud_rate.sort_values(ascending=False).head(3)
    
    print(f"\n{col}:")
    for cat, rate in top_fraud.items():
        count = train_df[(train_df[col]==cat) & (train_df['is_fraud']==1)].shape[0]
        print(f"  - {cat}: {count} fraud ({rate:.2f}% fraud rate)")

RINGKASAN: KATEGORI DENGAN FRAUD RATE TERTINGGI

gender:
  - M: 8327 fraud (14.15% fraud rate)
  - F: 5801 fraud (14.10% fraud rate)

country:
  - BR: 387 fraud (14.88% fraud rate)
  - UK: 1575 fraud (14.54% fraud rate)
  - DE: 209 fraud (14.30% fraud rate)

device_type:
  - mobile: 11791 fraud (15.06% fraud rate)
  - desktop: 1962 fraud (10.80% fraud rate)
  - tablet: 375 fraud (10.58% fraud rate)

device_os:
  - Windows: 2948 fraud (14.21% fraud rate)
  - MacOS: 250 fraud (14.18% fraud rate)
  - iOS: 3111 fraud (14.15% fraud rate)

merchant_category:
  - luxury: 180 fraud (16.23% fraud rate)
  - financial: 714 fraud (16.21% fraud rate)
  - travel: 1293 fraud (14.87% fraud rate)

transaction_type:
  - purchase: 8524 fraud (14.35% fraud rate)
  - topup: 2005 fraud (14.14% fraud rate)
  - withdrawal: 2266 fraud (13.69% fraud rate)


In [11]:
# ==============================================================================
# ANALISIS FITUR NUMERIK
# ==============================================================================
# 1. Apakah ada nilai ekstrem?
# 2. Apakah ada fitur yang sebenarnya "count" (nilai diskrit dengan sedikit unique values)?

# Identifikasi kolom numerik (exclude ID columns)
id_cols = ['ID', 'transaction_id', 'user_id']
numeric_cols = train_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols = [col for col in numeric_cols if col not in id_cols and col != 'is_fraud']

print("=" * 70)
print("ANALISIS FITUR NUMERIK")
print("=" * 70)
print(f"\nKolom numerik yang dianalisis: {numeric_cols}")
print(f"Jumlah kolom numerik: {len(numeric_cols)}")

ANALISIS FITUR NUMERIK

Kolom numerik yang dianalisis: ['age', 'transaction_amount', 'time_of_day', 'day_of_week', 'transaction_duration', 'num_prev_transactions', 'avg_transaction_amount', 'std_transaction_amount', 'transactions_last_24h', 'transactions_last_1h', 'failed_login_attempts', 'ip_risk_score', 'device_trust_score', 'account_age_days', 'has_chargeback_history', 'shared_ip_users', 'shared_device_users', 'merchant_risk', 'country_risk', 'distance_from_home', 'is_new_country']
Jumlah kolom numerik: 21


In [12]:
# ==============================================================================
# 1. CEK NILAI EKSTREM (OUTLIERS)
# ==============================================================================
# Menampilkan statistik deskriptif dan mendeteksi nilai ekstrem menggunakan IQR

print("=" * 70)
print("1. DETEKSI NILAI EKSTREM (OUTLIERS)")
print("=" * 70)

extreme_analysis = []

for col in numeric_cols:
    Q1 = train_df[col].quantile(0.25)
    Q3 = train_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Hitung outliers
    outliers_low = (train_df[col] < lower_bound).sum()
    outliers_high = (train_df[col] > upper_bound).sum()
    total_outliers = outliers_low + outliers_high
    outlier_pct = (total_outliers / len(train_df)) * 100
    
    extreme_analysis.append({
        'Column': col,
        'Min': train_df[col].min(),
        'Max': train_df[col].max(),
        'Mean': train_df[col].mean(),
        'Median': train_df[col].median(),
        'Std': train_df[col].std(),
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'Lower Bound': lower_bound,
        'Upper Bound': upper_bound,
        'Outliers Low': outliers_low,
        'Outliers High': outliers_high,
        'Total Outliers': total_outliers,
        'Outlier %': round(outlier_pct, 2)
    })

extreme_df = pd.DataFrame(extreme_analysis)

# Tampilkan kolom dengan outlier terbanyak
print("\n--- Statistik Outlier per Kolom (diurutkan berdasarkan % outlier) ---\n")
extreme_sorted = extreme_df.sort_values('Outlier %', ascending=False)
print(extreme_sorted[['Column', 'Min', 'Max', 'Mean', 'Median', 'Lower Bound', 'Upper Bound', 'Total Outliers', 'Outlier %']].to_string(index=False))

# Highlight kolom dengan outlier > 5%
print("\n" + "=" * 70)
print("KOLOM DENGAN OUTLIER > 5%:")
print("=" * 70)
high_outlier_cols = extreme_sorted[extreme_sorted['Outlier %'] > 5]
if len(high_outlier_cols) > 0:
    for _, row in high_outlier_cols.iterrows():
        print(f"\n{row['Column']}:")
        print(f"  - Range: [{row['Min']:.2f}, {row['Max']:.2f}]")
        print(f"  - Mean: {row['Mean']:.2f}, Median: {row['Median']:.2f}")
        print(f"  - IQR Bounds: [{row['Lower Bound']:.2f}, {row['Upper Bound']:.2f}]")
        print(f"  - Outliers: {row['Total Outliers']} ({row['Outlier %']}%)")
else:
    print("Tidak ada kolom dengan outlier > 5%")

1. DETEKSI NILAI EKSTREM (OUTLIERS)

--- Statistik Outlier per Kolom (diurutkan berdasarkan % outlier) ---

                Column    Min          Max        Mean   Median  Lower Bound  Upper Bound  Total Outliers  Outlier %
  transactions_last_1h   0.00    15.000000    0.309230    0.000       0.0000       0.0000           11697      11.70
 transactions_last_24h   0.00    44.000000    2.040150    1.000      -0.5000       3.5000            8878       8.88
   shared_device_users   0.00    11.000000    1.499800    1.000      -0.5000       3.5000            6612       6.61
has_chargeback_history   0.00     1.000000    0.060270    0.000       0.0000       0.0000            6027       6.03
    transaction_amount   0.01 20601.133350  126.855288   64.460    -129.3150     287.2850            5904       5.90
    distance_from_home   0.00  6388.517628  117.411584   71.810    -142.6100     316.6700            5550       5.55
        is_new_country   0.00     1.000000    0.049580    0.000       0.0

In [13]:
# ==============================================================================
# 2. CEK FITUR DENGAN "COUNT" KECIL (NILAI DISKRIT)
# ==============================================================================
# Fitur dengan sedikit unique values mungkin sebenarnya kategorikal/ordinal

print("=" * 70)
print("2. FITUR NUMERIK DENGAN UNIQUE VALUES SEDIKIT (POTENSI KATEGORIKAL)")
print("=" * 70)

count_analysis = []

for col in numeric_cols:
    n_unique = train_df[col].nunique()
    unique_vals = sorted(train_df[col].dropna().unique())
    
    # Tampilkan sample unique values (max 10)
    sample_vals = unique_vals[:10] if len(unique_vals) > 10 else unique_vals
    
    count_analysis.append({
        'Column': col,
        'Unique Values': n_unique,
        'Data Type': str(train_df[col].dtype),
        'Sample Values': sample_vals,
        'Is Integer-like': all(float(x).is_integer() for x in unique_vals if pd.notna(x))
    })

count_df = pd.DataFrame(count_analysis)

# Urutkan berdasarkan unique values
count_sorted = count_df.sort_values('Unique Values')

print("\n--- Semua Kolom Numerik (diurutkan berdasarkan jumlah unique values) ---\n")
for _, row in count_sorted.iterrows():
    print(f"{row['Column']}:")
    print(f"  - Unique values: {row['Unique Values']}")
    print(f"  - Data type: {row['Data Type']}")
    print(f"  - Integer-like: {row['Is Integer-like']}")
    if row['Unique Values'] <= 20:
        print(f"  - Semua nilai: {row['Sample Values']}")
    else:
        print(f"  - Sample (10 pertama): {row['Sample Values']}")
    print()

# Highlight kolom dengan unique values <= 20 (potensi kategorikal)
print("=" * 70)
print("KOLOM NUMERIK DENGAN UNIQUE VALUES <= 20 (POTENSI KATEGORIKAL/ORDINAL):")
print("=" * 70)
potential_categorical = count_sorted[count_sorted['Unique Values'] <= 20]
if len(potential_categorical) > 0:
    for _, row in potential_categorical.iterrows():
        print(f"\n{row['Column']}: {row['Unique Values']} unique values")
        print(f"  Nilai: {row['Sample Values']}")
else:
    print("\nTidak ada kolom numerik dengan unique values <= 20")

2. FITUR NUMERIK DENGAN UNIQUE VALUES SEDIKIT (POTENSI KATEGORIKAL)

--- Semua Kolom Numerik (diurutkan berdasarkan jumlah unique values) ---

has_chargeback_history:
  - Unique values: 2
  - Data type: int64
  - Integer-like: True
  - Semua nilai: [np.int64(0), np.int64(1)]

is_new_country:
  - Unique values: 2
  - Data type: int64
  - Integer-like: True
  - Semua nilai: [np.int64(0), np.int64(1)]

failed_login_attempts:
  - Unique values: 6
  - Data type: int64
  - Integer-like: True
  - Semua nilai: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

day_of_week:
  - Unique values: 7
  - Data type: int64
  - Integer-like: True
  - Semua nilai: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

country_risk:
  - Unique values: 7
  - Data type: float64
  - Integer-like: False
  - Semua nilai: [np.float64(0.02), np.float64(0.03), np.float64(0.04), np.float64(0.05), np.float64(0.1), np.float64(0.12), np.float64(0.13)]